# Data Preprocessing for 4 Crops: Maize, Rice, Cassava, Yam
This notebook cleans the NASA POWER climate data, aggregates it into sequence vectors, and merges it with crop yield data grouped by the 6 Geopolitical Zones in Nigeria.

## 1. Import Libraries & Load Data
This step imports necessary dependencies and reads the raw NASA climate records along with the raw crop production figures.

In [1]:
import pandas as pd
import numpy as np

# Load Data
climate_df = pd.read_csv('../../climate_data/data/combined/nasa_power_nigeria_all_1999_2023.csv')
crop_df = pd.read_csv('../../project_data/raw_data/crop_yield/adm_crop_production_NG.csv')

print("Loaded climate shape:", climate_df.shape)
print("Loaded crop shape:", crop_df.shape)


Loaded climate shape: (337847, 22)
Loaded crop shape: (27792, 16)


## 2. Filter Crops and Standardize Geopolitical Zones
Here we filter for our specific 4 crops (Maize, Rice, Cassava, Yam) and map individual states to their broader Geopolitical Zones to match the climate dataset. We also convert units from tonnes/ha to kg/ha for higher predictive granularity.

In [ ]:
# Target crops filtering (case insensitive equivalent logic)
target_crops = ['Yams', 'Cassava', 'Maize', 'Rice']
crop_filtered = crop_df[
    (crop_df['product'].isin(target_crops)) & 
    (crop_df['indicator'] == 'yield')
].copy()

# Fix 'Yams' to 'Yam'
crop_filtered['product'] = crop_filtered['product'].replace('Yams', 'Yam')

# Define state to zone mapping
state_to_zone = {
    'Benue': 'North-Central', 'Kogi': 'North-Central', 'Kwara': 'North-Central',
    'Nasarawa': 'North-Central', 'Niger': 'North-Central', 'Plateau': 'North-Central',
    'Federal Capital Territory': 'North-Central',
    'Adamawa': 'North-East', 'Bauchi': 'North-East', 'Borno': 'North-East',
    'Gombe': 'North-East', 'Taraba': 'North-East', 'Yobe': 'North-East',
    'Jigawa': 'North-West', 'Kaduna': 'North-West', 'Kano': 'North-West',
    'Katsina': 'North-West', 'Kebbi': 'North-West', 'Sokoto': 'North-West', 'Zamfara': 'North-West',
    'Abia': 'South-East', 'Anambra': 'South-East', 'Ebonyi': 'South-East',
    'Enugu': 'South-East', 'Imo': 'South-East',
    'Akwa Ibom': 'South-South', 'Bayelsa': 'South-South', 'Cross River': 'South-South',
    'Delta': 'South-South', 'Edo': 'South-South', 'Rivers': 'South-South',
    'Ekiti': 'South-West', 'Lagos': 'South-West', 'Ogun': 'South-West',
    'Ondo': 'South-West', 'Osun': 'South-West', 'Oyo': 'South-West'
}

crop_filtered['Geopolitical_Zone'] = crop_filtered['admin_1'].map(state_to_zone)

print("Crops:", crop_filtered['product'].unique())
print("Missed Mappings:", crop_filtered['Geopolitical_Zone'].isna().sum())

# Because yield is typically measured in tonnes/ha or kg/ha, let's just make sure it's numeric and clean missing.
crop_filtered.dropna(subset=['value'], inplace=True)
# We need to average by Year, Zone, and Crop when mapping upward from state to Zone level
crop_agg = crop_filtered.groupby(['planting_year', 'Geopolitical_Zone', 'product'])['value'].mean().reset_index()

crop_agg.rename(columns={
    'planting_year': 'Year', 
    'Geopolitical_Zone': 'Region', 
    'product': 'Crop',
    'value': 'Yield_kg_per_ha' # Renamed, will convert from t/ha to kg/ha
}, inplace=True)

# Convert from t/ha to kg/ha
crop_agg['Yield_kg_per_ha'] = crop_agg['Yield_kg_per_ha'] * 1000

print("Aggregated crop yield data across regions:", crop_agg.shape)


Crops: ['Yam' 'Cassava' 'Maize' 'Rice']
Missed Mappings: 0
Aggregated crop yield data across regions: (600, 4)


## 3. Aggregate Climate Data to Monthly Intervals
NASA POWER climate data comes daily. This step rolls the daily features (like average temperature and total precipitation) into distinct, sequenced monthly averages and sums to form our required 12-month temporal array.

In [4]:
# Aggregate climate daily features Up to monthly, per Region/Year
# Note: NASA POWER already contains 'Geopolitical_Zone'
climate_features = ['T2M', 'T2M_MAX', 'T2M_MIN', 'TS', 'T2MDEW', 'T2MWET', 
                    'PRECTOTCORR', 'RH2M', 'QV2M', 'GWETTOP', 'GWETROOT', 
                    'WS2M', 'PS', 'ALLSKY_SFC_SW_DWN', 'CLRSKY_SFC_SW_DWN', 
                    'ALLSKY_SFC_LW_DWN', 'CLOUD_AMT']
                    
agg_dict = {feat: 'mean' for feat in climate_features}
# Exception for precipitation (usually Sum for totals, but typically mean is daily average so mean works for consistent scales or sum is fine. We will sum PRECTOTCORR)
agg_dict['PRECTOTCORR'] = 'sum'

# Ensure Date and Month are handled
climate_monthly = climate_df.groupby(['Geopolitical_Zone', 'Year', 'Month']).agg(agg_dict).reset_index()

# Pivot to wide format: (Month 1, Month 2... Month 12 for each feature)
climate_wide = climate_monthly.pivot_table(
    index=['Geopolitical_Zone', 'Year'],
    columns='Month',
    values=climate_features
)

# Flatten columns to feature_m1, feature_m2, etc.
climate_wide.columns = [f"{feat}_m{int(month)}" for feat, month in climate_wide.columns]
climate_wide = climate_wide.reset_index()

print("Pivoted climate shape:", climate_wide.shape)


Pivoted climate shape: (150, 206)


## 4. Merge Crop and Climate Data & Export
Finally, we perform an inner join between the aggregated yearly crop yields and the unrolled month-by-month climate sequence targets, saving the result safely out to `processed_dataset.csv`.

In [ ]:
climate_wide.rename(columns={'Geopolitical_Zone': 'Region'}, inplace=True)
merged_df = pd.merge(crop_agg, climate_wide, on=['Region', 'Year'], how='inner')

print("Merged Dataframe shape:", merged_df.shape)
print("Crop counts:", merged_df['Crop'].value_counts())

# Save to `New_Changes/data/processed_dataset.csv`
out_path = '../data/processed_dataset.csv'
merged_df.to_csv(out_path, index=False)
print(f"Data saved to {out_path}!")

Merged Dataframe shape: (600, 208)
Crop counts: Crop
Cassava    150
Maize      150
Rice       150
Yam        150
Name: count, dtype: int64
Data saved to ../data/processed_dataset.csv!
